# Mechanistic Interpretability Battery

This comprehensive notebook executes the full suite of Mechanistic Interpretability techniques on the trained World Model. It runs through:
1. Linear Probing
2. Logit Lens
3. Activation Patching
4. Attention Head Analysis
5. Direct Logit Attribution & Graphs
6. Sparse Autoencoders (SAEs)
7. Developmental Interpretability (Grokking tracking)

## Stage 1: Linear Probing Sweep
Extract intermediate layer representations and run probes to find out where the model stores concepts like Registers and Flags.

In [ ]:
import jsonlines
from interp.probing.probe_sweep import sweep_all_layers, plot_probe_curves

records = []
with jsonlines.open("data/splits/phase2_raw.jsonl") as reader:
    for r in reader:
        if r["split"] == "test_indist":
            records.append(r)
            if len(records) > 200: break

activation_cache = {} # Populate with actual activations from the model

# df = sweep_all_layers(activation_cache, records, n_layers=24)
# plot_probe_curves(df)

## Stage 2: Logit Lens
Un-embeds intermediate layers to vocabulary space to reveal how the model's 'belief' about the final target token evolves across depth.

In [ ]:
from interp.logit_lens.logit_lens import compute_logit_lens
from interp.logit_lens.visualise import plot_logit_lens_heatmap

prompt = "[PC=001] [R1=00000] [R2=00001] [FLAG=N]"

# results = compute_logit_lens("Qwen/Qwen2.5-0.5B-Instruct", prompt, top_k=5)
# plot_logit_lens_heatmap(results, target_position=-1, title="Logit Lens on MIS State Token")

## Stage 3: Activation Patching
Selectively swap activations from a 'corrupted' prompt to a 'clean' prompt to pinpoint causal mediators.

In [ ]:
from interp.patching.activation_patcher import run_activation_patch_sweep, plot_patching_heatmap

clean_prompt = "[PC=002] ADD R1 R2"
corrupt_prompt = "[PC=002] SUB R1 R2"

# effect_matrix = run_activation_patch_sweep(
#     model_path="Qwen/Qwen2.5-0.5B-Instruct",
#     clean_prompt=clean_prompt,
#     corrupted_prompt=corrupt_prompt,
#     target_token="R3",
#     n_layers=24,
#     n_tokens=10
# )
# plot_patching_heatmap(effect_matrix, token_labels=[f"T{i}" for i in range(10)])

## Stage 4: Attention Head Analysis
Visualize attention patterns to discover induction heads or 'operand retrieval heads' attending to variables.

In [ ]:
from interp.attention.head_analyser import extract_attention_patterns, score_operand_retrieval

prompts = ["LOAD #5 -> R1 \n ADD R1 R1 -> R2"]
# patterns = extract_attention_patterns("Qwen/Qwen2.5-0.5B-Instruct", prompts, n_layers=24, n_heads=16)

# scores = score_operand_retrieval(patterns, instr_positions=[6], src_positions=[(3, 4)])
# print(f"Operand Retrieval Score: {scores.max():.2f} at layer {scores.argmax()//16}, head {scores.argmax()%16}")

## Stage 5: Direct Logit Attribution (DLA) & Attribution Graphs
Score how individual heads/MLPs push the output logit directly, and build attribution graphs.

In [ ]:
from interp.attribution.direct_logit_attribution import compute_dla
from interp.attribution.attribution_graph import build_simplified_attribution_graph, visualise_attribution_graph
import plotly.express as px

# dla_results = compute_dla(model, cache, target_id, 24, 16, 1024)
# px.imshow(dla_results['attn_heads'], title="DLA by Attention Head (Layer x Head)").show()

# G = build_simplified_attribution_graph(model, "ADD R1 R2 -> R3", target_token_id=544, n_layers=24, n_heads=16)
# visualise_attribution_graph(G, output_path="attribution_graph.html")

## Stage 6: Sparse Autoencoders (SAEs)
Extract human-interpretable features out of embeddings using `saelens`.

In [ ]:
from interp.sae.train_sae import train_sae_on_layer
from interp.sae.analyse_features import find_top_activating_examples

# sae = train_sae_on_layer("Qwen/Qwen2.5-0.5B-Instruct", "", target_layer=10, d_model=1024)
# analysis = find_top_activating_examples(sae, cache, records, feature_idx=42, layer=10)
# print(analysis)

## Stage 7: Developmental Interpretability (Grokking)
Track linear probes over training time to look for phase changes (Grokking).

In [ ]:
from interp.developmental.checkpoint_tracker import track_probes_across_checkpoints, plot_developmental_curves

def dummy_cache_fn(ckpt_path):
    return {}

# df = track_probes_across_checkpoints("checkpoints/phase2_qwen0.5b_condB", dummy_cache_fn, records=[])
# plot_developmental_curves(df)